In [5]:
pip install mlflow


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 24.7 MB 81.7 MB/s eta 0:00:01
     |████████████████████████████████| 147 kB 48.9 MB/s eta 0:00:01
     |████████████████████████████████| 103 kB 40.2 MB/s eta 0:00:01
     |████████████████████████████████| 7.9 MB 35.8 MB/s eta 0:00:01
     |████████████████████████████████| 247 kB 34.4 MB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 23.7 MB/s eta 0:00:01
     |████████████████████████████████| 12.1 MB 53.4 MB/s eta 0:00:01
     |████████████████████████████████| 85 kB 10.9 MB/s eta 0:00:01
     |████████████████████████████████| 2.1 MB 28.4 MB/s eta 0:00:01
     |████████████████████████████████| 39.4 MB 37.2 MB/s eta 0:00:01
     |████████████████████████████████| 114 kB 45.4 MB/s eta 0:00:01
     |████████████████████████████████| 32.3 MB 63.5 MB/s eta 0:00:01    |██▋                             | 2.6 MB 46.2 MB/s eta 0:00:01     |███████████████████

In [6]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
import pandas as pd

/Users/mj_peace/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
filepath = '/Users/mj_peace/Desktop/MLOPS/Datasets/heart.csv'
data = pd.read_csv(filepath)
data.head()


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score
from sklearn.ensemble import RandomForestClassifier

# supervise machine learning classfication model 
X = data.drop('target', axis=1)
y = data['target']
# Since RandomForestRegressor is a regression model, we need to convert its predictions to binary for classification metrics
# Predict and threshold at 0.5
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

n_estimators = 5
max_depth=4
random_state=37
model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=random_state)
model.fit(X_test, y_test)
y_pred = model.predict(X_test)

precision = precision_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)


print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Precision:", precision)


# Log the model with MLflow
mlflow.set_experiment("heart_disease_prediction_rf")

with mlflow.start_run():
    mlflow.sklearn.log_model(model, "model")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("precision", precision)
    mlflow.sklearn.log_model(model, "random_forest_classifier")
    print("Model logged with MLflow")
    print("Run ID:", mlflow.active_run().info.run_id)
    print(f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, F1 Score: {f1:.4f}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Train a Decision Tree Classifier
dt_max_depth = 5
dt_random_state = 37
dt_model = DecisionTreeClassifier(max_depth=dt_max_depth, random_state=dt_random_state)
dt_model.fit(X_train, y_train)
dt_y_pred = dt_model.predict(X_test)

dt_precision = precision_score(y_test, dt_y_pred)
dt_accuracy = accuracy_score(y_test, dt_y_pred)
dt_f1 = f1_score(y_test, dt_y_pred)

print("Decision Tree Accuracy:", dt_accuracy)
print("Decision Tree F1 Score:", dt_f1)
print("Decision Tree Precision:", dt_precision)

# Log the Decision Tree model with MLflow
mlflow.set_experiment("heart_disease_prediction_decision_tree")

with mlflow.start_run():
    mlflow.sklearn.log_model(dt_model, "decision_tree_classifier")
    mlflow.log_param("max_depth", dt_max_depth)
    mlflow.log_param("random_state", dt_random_state)
    mlflow.log_metric("accuracy", dt_accuracy)
    mlflow.log_metric("f1_score", dt_f1)
    mlflow.log_metric("precision", dt_precision)
    print("Decision Tree model logged with MLflow")
    print("Run ID:", mlflow.active_run().info.run_id)
    print(f"Accuracy: {dt_accuracy:.4f}, Precision: {dt_precision:.4f}, F1 Score: {dt_f1:.4f}")

In [ ]:
# ============================================
# Load model from MLflow Experiment and Predict
# ============================================

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

# 1. Tracking URI must be the same as training
mlflow.set_tracking_uri("file:./mlruns")


# 2. Replace with your experiment name or ID
experiment_name = "1708_xgboost_mlops_heart_case"
exp = mlflow.get_experiment_by_name(experiment_name)
print(exp)
experiment_id = exp.experiment_id
print("Using Experiment ID:", experiment_id)

# 3. Get latest run from this experiment
client = MlflowClient()
runs = client.search_runs(experiment_id, order_by=["attributes.start_time desc"], max_results=1)

if not runs:
    raise Exception("No runs found in experiment:", experiment_name)

run = runs[0]
run_id = run.info.run_id
print("Latest Run ID:", run_id)

# 4. Construct model path
model_uri = f"runs:/{run_id}/xgboost_classifier"
print("Loading model from:", model_uri)

# 5. Load model
model = mlflow.xgboost.load_model(model_uri)

# 6. Run predictions on X_test (must reuse the same X_test as in training script)
y_pred = model.predict(X_test)

print("Predictions:", y_pred[:10])
print("True Labels:", y_test[:10])

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier

# Prepare data (reuse X_train, X_test, y_train, y_test from previous cells)
xgb_model = XGBClassifier(n_estimators=100, max_depth=4, random_state=37, use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

# Calculate metrics
xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb)

print("XGBoost Accuracy:", xgb_accuracy)
print("XGBoost F1 Score:", xgb_f1)
print("XGBoost Precision:", xgb_precision)

# Log the XGBoost model with MLflow
mlflow.set_experiment("heart_disease_prediction_xgboost")

with mlflow.start_run():
    mlflow.sklearn.log_model(xgb_model, "xgboost_classifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("random_state", 37)
    mlflow.log_metric("accuracy", xgb_accuracy)
    mlflow.log_metric("f1_score", xgb_f1)
    mlflow.log_metric("precision", xgb_precision)
    print("XGBoost model logged with MLflow")
    print("Run ID:", mlflow.active_run().info.run_id)
    print(f"Accuracy: {xgb_accuracy:.4f}, Precision: {xgb_precision:.4f}, F1 Score: {xgb_f1:.4f}")

In [ ]:
import subprocess
import time

# Start the MLflow UI on port 5000
subprocess.Popen(["mlflow", "ui", "--port", "5000"])

# Optional: Give it time to start
time.sleep(3)

print("MLflow UI running at http://localhost:5000")

MLflow UI running at http://localhost:5000


[2025-08-22 21:33:01 -0500] [37566] [INFO] Starting gunicorn 23.0.0
[2025-08-22 21:33:01 -0500] [37566] [ERROR] Connection in use: ('127.0.0.1', 5000)
[2025-08-22 21:33:01 -0500] [37566] [ERROR] connection to ('127.0.0.1', 5000) failed: [Errno 48] Address already in use
[2025-08-22 21:33:02 -0500] [37566] [ERROR] Connection in use: ('127.0.0.1', 5000)
[2025-08-22 21:33:02 -0500] [37566] [ERROR] connection to ('127.0.0.1', 5000) failed: [Errno 48] Address already in use
[2025-08-22 21:33:03 -0500] [37566] [ERROR] Connection in use: ('127.0.0.1', 5000)
[2025-08-22 21:33:03 -0500] [37566] [ERROR] connection to ('127.0.0.1', 5000) failed: [Errno 48] Address already in use
[2025-08-22 21:33:04 -0500] [37566] [ERROR] Connection in use: ('127.0.0.1', 5000)
[2025-08-22 21:33:04 -0500] [37566] [ERROR] connection to ('127.0.0.1', 5000) failed: [Errno 48] Address already in use
[2025-08-22 21:33:05 -0500] [37566] [ERROR] Connection in use: ('127.0.0.1', 5000)
[2025-08-22 21:33:05 -0500] [37566] [